# Student Health Risk Prediction

## Competition Understanding and Dataset Verification

**Module:** CIS6005 Computational Intelligence  
**Competition:** Predicting Student Health Risk  
**Competition Series:** Kaggle Playground Series - Season 6, Episode 7  

### Project Aim

The aim of this project is to develop and compare multiple machine learning and computational intelligence models for predicting student health risk categories. Four classification approaches are investigated:

1. Logistic Regression
2. Random Forest
3. Support Vector Machine
4. Multi-Layer Perceptron Neural Network

The best-performing model will be selected through systematic evaluation and used to generate predictions for the Kaggle competition dataset. The selected trained model will subsequently be integrated into a usable web application.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "test.csv"
SUBMISSION_PATH = PROJECT_ROOT / "data" / "raw" / "sample_submission.csv"

print("Project Root:", PROJECT_ROOT)
print("Train Path:", TRAIN_PATH)
print("Test Path:", TEST_PATH)
print("Submission Path:", SUBMISSION_PATH)

Project Root: /Users/admin/Documents/ICBT/Computational Intelligence/Student Health Risk Prediction/student-health-risk-prediction
Train Path: /Users/admin/Documents/ICBT/Computational Intelligence/Student Health Risk Prediction/student-health-risk-prediction/data/raw/train.csv
Test Path: /Users/admin/Documents/ICBT/Computational Intelligence/Student Health Risk Prediction/student-health-risk-prediction/data/raw/test.csv
Submission Path: /Users/admin/Documents/ICBT/Computational Intelligence/Student Health Risk Prediction/student-health-risk-prediction/data/raw/sample_submission.csv


In [3]:
print("Train file exists:", TRAIN_PATH.exists())
print("Test file exists:", TEST_PATH.exists())
print("Submission file exists:", SUBMISSION_PATH.exists())

Train file exists: True
Test file exists: True
Submission file exists: True


In [4]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission_df = pd.read_csv(SUBMISSION_PATH)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [5]:
print("Train Dataset Shape:", train_df.shape)
print("Test Dataset Shape:", test_df.shape)
print("Sample Submission Shape:", sample_submission_df.shape)

Train Dataset Shape: (690088, 15)
Test Dataset Shape: (295753, 14)
Sample Submission Shape: (295753, 2)


### Dataset Size Interpretation

The training dataset contains labelled observations that can be used for model development and local validation. The test dataset contains unlabelled observations for which predictions must be generated. The sample submission file defines the structure required for a valid Kaggle submission.

The relatively large dataset size has important implications for model design. Computationally efficient approaches are required, particularly for Support Vector Machine training, because conventional kernel-based SVM methods may become expensive in terms of execution time and memory consumption on large datasets.

In [6]:
train_df.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [7]:
test_df.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [8]:
sample_submission_df.head()

,id,health_condition
0,690088,at-risk
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,at-risk


In [9]:
print("Training Columns:")
print(train_df.columns.tolist())

print("\nTest Columns:")
print(test_df.columns.tolist())

print("\nSubmission Columns:")
print(sample_submission_df.columns.tolist())

Training Columns:
['id', 'health_condition', 'sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

Test Columns:
['id', 'sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

Submission Columns:
['id', 'health_condition']


In [10]:
train_columns = set(train_df.columns)
test_columns = set(test_df.columns)

train_only_columns = train_columns - test_columns
test_only_columns = test_columns - train_columns

print("Columns only in training data:", train_only_columns)
print("Columns only in test data:", test_only_columns)

Columns only in training data: {'health_condition'}
Columns only in test data: set()


In [11]:
target_candidates = [
    col for col in train_df.columns
    if col not in test_df.columns
]

print("Possible Target Column(s):", target_candidates)

Possible Target Column(s): ['health_condition']


In [12]:
TARGET = "health_condition"
ID_COLUMN = "id"

print("Target Column:", TARGET)
print("ID Column:", ID_COLUMN)

Target Column: health_condition
ID Column: id


In [13]:
print("Unique Target Classes:")
print(train_df[TARGET].unique())

print("\nNumber of Target Classes:")
print(train_df[TARGET].nunique())

Unique Target Classes:
<StringArray>
['unhealthy', 'at-risk', 'fit']
Length: 3, dtype: str

Number of Target Classes:
3


In [14]:
target_counts = train_df[TARGET].value_counts()
target_percentages = (
    train_df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percentages
})

target_summary

,Count,Percentage
health_condition,,
at-risk,592561,85.87
unhealthy,57724,8.36
fit,39803,5.77


### Target Variable Interpretation

The target variable is `health_condition`, indicating that the task is a supervised classification problem. The distribution of target classes should be examined carefully because substantial class imbalance can cause overall accuracy to provide an overly optimistic representation of model performance.

Therefore, the project will consider additional evaluation measures such as balanced accuracy, macro-averaged precision, macro-averaged recall, macro F1-score, class-wise recall, and confusion matrices. The official Kaggle competition metric will also be considered when evaluating leaderboard-oriented performance.

In [15]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  str    
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  str    
 10  stress_level             607277 non-null  str    
 11  sleep_quality            631757 non-null  str    
 12  physical_activity_level  653467 non-null  str    
 13  smoking_alcohol          661506 non-null  str    
 14  gender         

In [16]:
dtype_summary = pd.DataFrame({
    "Column": train_df.columns,
    "Data Type": train_df.dtypes.astype(str).values,
    "Non-Null Count": train_df.notnull().sum().values,
    "Null Count": train_df.isnull().sum().values
})

dtype_summary

,Column,Data Type,Non-Null Count,Null Count
0,id,int64,690088,0
1,health_condition,str,690088,0
2,sleep_duration,float64,614089,75999
3,heart_rate,float64,682255,7833
4,bmi,float64,676190,13898
5,calorie_expenditure,float64,637235,52853
6,step_count,float64,676172,13916
7,exercise_duration,float64,683187,6901
8,water_intake,float64,646611,43477
9,diet_type,str,683187,6901


In [17]:
numerical_features = train_df.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = train_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical Columns:")
print(numerical_features)

print("\nCategorical Columns:")
print(categorical_features)

Numerical Columns:
['id', 'sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

Categorical Columns:
['health_condition', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [18]:
numerical_model_features = [
    col for col in numerical_features
    if col not in [ID_COLUMN, TARGET]
]

categorical_model_features = [
    col for col in categorical_features
    if col != TARGET
]

print("Numerical Model Features:")
print(numerical_model_features)

print("\nCategorical Model Features:")
print(categorical_model_features)

Numerical Model Features:
['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

Categorical Model Features:
['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [19]:
missing_summary = pd.DataFrame({
    "Missing Count": train_df.isnull().sum(),
    "Missing Percentage": (
        train_df.isnull().mean() * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values(
    by="Missing Percentage",
    ascending=False
)

missing_summary

,Missing Count,Missing Percentage
stress_level,82811,12.00
sleep_duration,75999,11.01
sleep_quality,58331,8.45
calorie_expenditure,52853,7.66
water_intake,43477,6.30
physical_activity_level,36621,5.31
smoking_alcohol,28582,4.14
gender,21373,3.10
step_count,13916,2.02
bmi,13898,2.01


In [20]:
total_missing = train_df.isnull().sum().sum()
total_cells = train_df.shape[0] * train_df.shape[1]
missing_percentage = (total_missing / total_cells) * 100

print("Total Missing Values:", total_missing)
print("Overall Missing Percentage:", round(missing_percentage, 2), "%")

Total Missing Values: 449496
Overall Missing Percentage: 4.34 %


In [21]:
duplicate_count = train_df.duplicated().sum()

print("Number of Duplicate Rows:", duplicate_count)

Number of Duplicate Rows: 0


In [22]:
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,690088.0,345043.500000,199211.390620,0.0,172521.75,345043.50,517565.25,690087.00
sleep_duration,614089.0,6.992597,1.215407,3.0,6.16,6.99,7.81,10.00
heart_rate,682255.0,75.096504,8.175106,50.0,69.40,75.10,80.70,107.70
bmi,676190.0,22.984925,2.481787,16.0,21.32,22.99,24.66,34.82
calorie_expenditure,637235.0,2226.084931,347.532098,1200.0,2053.00,2241.00,2456.00,3580.00
step_count,676172.0,8615.953050,3929.399831,1002.0,5389.00,8856.00,12114.00,14999.00
exercise_duration,683187.0,38.751456,14.742189,0.0,29.20,39.40,49.40,99.80
water_intake,646611.0,2.188542,0.518489,0.5,1.84,2.17,2.50,4.72


In [23]:
if categorical_model_features:
    categorical_summary = pd.DataFrame({
        "Feature": categorical_model_features,
        "Unique Values": [
            train_df[col].nunique(dropna=True)
            for col in categorical_model_features
        ],
        "Missing Values": [
            train_df[col].isnull().sum()
            for col in categorical_model_features
        ]
    })

    display(categorical_summary)
else:
    print("No categorical model features detected.")

,Feature,Unique Values,Missing Values
0,diet_type,3,6901
1,stress_level,3,82811
2,sleep_quality,3,58331
3,physical_activity_level,3,36621
4,smoking_alcohol,3,28582
5,gender,3,21373


In [24]:
train_features = train_df.drop(columns=[TARGET])
test_features = test_df.copy()

print("Training Feature Shape:", train_features.shape)
print("Test Feature Shape:", test_features.shape)

print(
    "Feature columns match:",
    list(train_features.columns) == list(test_features.columns)
)

Training Feature Shape: (690088, 14)
Test Feature Shape: (295753, 14)
Feature columns match: True


In [25]:
print("Test Rows:", len(test_df))
print("Submission Rows:", len(sample_submission_df))

print(
    "Row counts match:",
    len(test_df) == len(sample_submission_df)
)

Test Rows: 295753
Submission Rows: 295753
Row counts match: True


In [26]:
if ID_COLUMN in test_df.columns and ID_COLUMN in sample_submission_df.columns:
    ids_match = test_df[ID_COLUMN].equals(
        sample_submission_df[ID_COLUMN]
    )

    print("Test and submission IDs match:", ids_match)
else:
    print("ID column not found in both datasets.")

Test and submission IDs match: True


In [27]:
models = {
    "Logistic Regression": {
        "category": "Linear Classification",
        "role": "Interpretable baseline model"
    },

    "Random Forest": {
        "category": "Ensemble Learning",
        "role": "Non-linear ensemble classifier"
    },

    "Support Vector Machine": {
        "category": "Kernel / Margin-based Learning",
        "role": "Complex decision boundary modelling"
    },

    "MLP Neural Network": {
        "category": "Neural Network",
        "role": "Non-linear representation learning"
    }
}

model_plan_df = pd.DataFrame(models).T
model_plan_df

,category,role
Logistic Regression,Linear Classification,Interpretable baseline model
Random Forest,Ensemble Learning,Non-linear ensemble classifier
Support Vector Machine,Kernel / Margin-based Learning,Complex decision boundary modelling
MLP Neural Network,Neural Network,Non-linear representation learning


In [28]:
evaluation_metrics = [
    "Accuracy",
    "Balanced Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1-score",
    "Class-wise Precision",
    "Class-wise Recall",
    "Class-wise F1-score",
    "Confusion Matrix",
    "Official Kaggle Competition Metric"
]

pd.DataFrame({
    "Planned Evaluation Metric": evaluation_metrics
})

,Planned Evaluation Metric
0,Accuracy
1,Balanced Accuracy
2,Macro Precision
3,Macro Recall
4,Macro F1-score
5,Class-wise Precision
6,Class-wise Recall
7,Class-wise F1-score
8,Confusion Matrix
9,Official Kaggle Competition Metric


## Initial Competition and Dataset Findings

This notebook established the technical foundation for the student health risk prediction project. The problem is formulated as a supervised classification task in which the target variable is predicted from student health and lifestyle-related features.

The initial inspection identified the dataset dimensions, feature types, target classes, missing-value patterns, duplicate records, and compatibility between the training and test datasets. These findings will directly influence the subsequent preprocessing and modelling strategy.

The project will compare four models: Logistic Regression, Random Forest, Support Vector Machine, and a Multi-Layer Perceptron Neural Network. Logistic Regression will provide an interpretable baseline, Random Forest will represent ensemble learning, SVM will investigate margin-based non-linear classification, and the MLP will evaluate the suitability of neural computation for the tabular prediction problem.

Particular attention will be given to missing-data treatment, feature scaling, categorical encoding, class-sensitive evaluation, computational cost, and model generalisation. Because different algorithms have different preprocessing requirements, model-specific pipelines will be developed rather than applying an identical transformation strategy without justification.

The next stage of the project is a detailed Exploratory Data Analysis to investigate distributions, class imbalance, feature relationships, outliers, and patterns that may influence model design.